In [115]:
import pandas as pd
import numpy as np

In [116]:
def get_engineers_info(control_df: pd.DataFrame) -> pd.DataFrame:
    _start = pd.to_datetime(control_df["Начало"], format="%d.%m.%Y %H:%M")
    _end = pd.to_datetime(control_df["Окончание"], format="%d.%m.%Y %H:%M")

    agg_kwargs = {
        "shift_start": ("_start", "min"),
        "shift_end": ("_end", "max"),
        "gigabit_connection": ("Гигабитное подключение", lambda s: "Да" in set(s.dropna().unique())),
    }

    if "Подключение" in control_df.columns:
        agg_kwargs["equipment_types"] = (
            "Подключение",
            lambda s: set(s.dropna().unique())
        )

    engineers_df = (
        control_df.assign(_start=_start, _end=_end)
        .dropna(subset=["Бригада"])
        .groupby("Бригада")
        .agg(**agg_kwargs)
        .reset_index()
        .sort_values("shift_start")
    )

    engineers_df["shift_start"] = engineers_df["shift_start"].dt.time
    engineers_df["shift_end"] = engineers_df["shift_end"].dt.time

    if "equipment_types" not in engineers_df.columns:
        engineers_df["equipment_types"] = [set() for _ in range(len(engineers_df))]

    def _build_equipment(row):
        equipment = set(row["equipment_types"]) if row["equipment_types"] else set()
        if row["gigabit_connection"]:
            equipment.add("gigabit_connection")
        return equipment or None

    engineers_df["equipment"] = engineers_df.apply(_build_equipment, axis=1)
    engineers_df = engineers_df.drop(columns=["equipment_types", "gigabit_connection"])

    return engineers_df

In [117]:
east_synthetic_df = pd.read_csv("./data/Восток Синтетические данные.csv", sep=';', encoding="windows-1251")
east_control_df = pd.read_csv("./data/Восток Контрольное распределение..csv", sep=';', encoding="windows-1251")

In [118]:
GROUP_KEYS = ["Начало", "Окончание", "Район"]

REQ_TO_SKILL_COLUMN = {
    "required_skill_local_works": "skill_local_works",
    "required_skill_connection_works": "skill_connection_works",
    "required_skill_emergency_works": "skill_emergency_works",
}


def assign_required_skills(requests_df: pd.DataFrame, p_true: float = 0.5, seed: int | None = None) -> pd.DataFrame:
    """Fill required_skill_* with independent random booleans."""
    rng = np.random.default_rng(seed)
    requests_df = requests_df.copy()
    n = len(requests_df)

    for col in REQ_TO_SKILL_COLUMN:
        requests_df[col] = rng.random(n) < p_true

    return requests_df


def attach_skills_to_control(control_df: pd.DataFrame, requests_df: pd.DataFrame) -> pd.DataFrame:
    if len(control_df) != len(requests_df):
        raise ValueError(f"{len(control_df)} rows in control_df vs {len(requests_df)} in requests_df.")

    control_df = control_df.reset_index(drop=True).copy()
    requests_reset = requests_df.reset_index(drop=True)

    # NaN != NaN in pandas, so fill with a sentinel before comparing —
    # otherwise "both missing" gets flagged as a mismatch
    _SENTINEL = "__MISSING__"
    control_keys = control_df[GROUP_KEYS].fillna(_SENTINEL)
    requests_keys = requests_reset[GROUP_KEYS].fillna(_SENTINEL)

    diff_mask = control_keys != requests_keys
    mismatches = diff_mask.any(axis=1)
    if mismatches.any():
        bad_idx = mismatches[mismatches].index.tolist()
        print(f"Row mismatch at {len(bad_idx)} position(s), showing up to 5:")
        for i in bad_idx[:5]:
            bad_cols = diff_mask.columns[diff_mask.loc[i]].tolist()
            for col in bad_cols:
                print(
                    f"  row {i} | column {col!r}: control={control_df.loc[i, col]!r} vs requests={requests_reset.loc[i, col]!r}")
        raise ValueError(f"Row mismatch at position(s) {bad_idx[:5]} — control_df and requests_df aren't aligned.")

    for req_col, skill_col in REQ_TO_SKILL_COLUMN.items():
        control_df[skill_col] = requests_reset[req_col]

    return control_df


def fill_engineer_skills(engineers_df: pd.DataFrame, control_with_skills: pd.DataFrame) -> pd.DataFrame:
    """A brigade 'has' a skill if any ticket they were assigned required it."""
    engineers_df = engineers_df.copy()
    skill_cols = list(REQ_TO_SKILL_COLUMN.values())

    skill_by_brigade = (
        control_with_skills.dropna(subset=["Бригада"])
        .groupby("Бригада")[skill_cols]
        .any()
    )

    engineers_df = engineers_df.merge(skill_by_brigade, left_on="Бригада", right_index=True, how="left")
    for col in skill_cols:
        engineers_df[col] = engineers_df[col].fillna(False)

    def _build_skills(row):
        skills = set()
        for col in skill_cols:
            if row[col]:
                skills.add(col)
        return skills

    engineers_df["skills"] = engineers_df.apply(_build_skills, axis=1)
    engineers_df = engineers_df.drop(columns=skill_cols)

    return engineers_df

In [119]:
east_synthetic_df = assign_required_skills(east_synthetic_df, seed=42)
east_control_with_skills = attach_skills_to_control(east_control_df, east_synthetic_df)
engineers_east_df = fill_engineer_skills(get_engineers_info(east_control_with_skills), east_control_with_skills)
engineers_east_df["office"] = "г. Москва, ул Юных Ленинцев, д 83с 4"

In [120]:
engineers_east_df.to_csv("./data/EastEngineersData.csv", index=False)
east_synthetic_df.to_csv("./data/EastSyntethicData.csv", index=False)
east_control_with_skills.to_csv("./data/EastControlData.csv", index=False)

In [121]:
south_east_synthetic_df = pd.read_csv("./data/Юго-восток Синтетические данные.csv", sep=';', encoding="windows-1251")
south_east_control_df = pd.read_csv("./data/Юго-восток Контрольное распределение.csv", sep=';', encoding="windows-1251")

In [122]:
south_east_synthetic_df = assign_required_skills(south_east_synthetic_df, seed=42)
south_east_control_with_skills = attach_skills_to_control(south_east_control_df, south_east_synthetic_df)
south_east_engineers_df = fill_engineer_skills(get_engineers_info(south_east_control_with_skills),
                                               south_east_control_with_skills)
south_east_engineers_df["office"] = "г. Москва, ул Бирюлёвская, д 1с1"

In [123]:
south_east_engineers_df.to_csv("./data/SouthEastEngineersData.csv", index=False)
south_east_synthetic_df.to_csv("./data/SouthEastSyntethicData.csv", index=False)
south_east_control_with_skills.to_csv("./data/SouthEastControlData.csv", index=False)

In [124]:
south_center_synthetic_df = pd.read_csv("./data/ЮгоЦентр Синтетические данные.csv", sep=';', encoding="windows-1251")
south_center_control_df = pd.read_csv("./data/Югоцентр Контрольное распределение..csv", sep=';',
                                      encoding="windows-1251")

In [125]:
south_center_synthetic_df = assign_required_skills(south_center_synthetic_df, seed=42)
south_center_control_with_skills = attach_skills_to_control(south_center_control_df, south_center_synthetic_df)
south_center_east_engineers_df = fill_engineer_skills(get_engineers_info(south_center_control_with_skills),
                                                      south_center_control_with_skills)
south_center_east_engineers_df["office"] = "г.Москва проезд Симферопольский, д.7"

In [126]:
south_center_synthetic_df.to_csv("./data/SouthCenterSyntethicData.csv", index=False)
south_center_control_with_skills.to_csv("./data/SouthCenterControlData.csv", index=False)
south_center_east_engineers_df.to_csv("./data/SouthCenterEngineersData.csv", index=False)